# Install required libraries
Run the following to install core RAG + LangChain dependencies.
You can run the install cell below inside this notebook or run the same commands in your terminal.

In [1]:
# Install packages (run in notebook or terminal)
!python -m pip install --upgrade pip
#!python -m pip install langchain openai tiktoken sentence-transformers chromadb faiss-cpu
# Note: On Windows, installing faiss-cpu may fail with pip. If that happens,
# use conda: conda install -c conda-forge faiss-cpu

In [2]:
pip install -r requirements.txt

  Using cached jsonpointer-3.0.0-py2.py3-none-any.whl.metadata (2.3 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Using cached anyio-4.12.0-py3-none-any.whl.metadata (4.3 kB)
  Using cached httpcore-1.0.9-py3-none-any.whl.metadata (21 kB)
  Using cached idna-3.11-py3-none-any.whl.metadata (8.4 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached numpy-2.2.6-cp310-cp310-win_amd64.whl.metadata (60 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached overrides-7.7.0-py3-none-any.whl.metadata (5.8 kB)
  Using cached importlib_resources-6.5.2-py3-none-any.whl.metadata (3.9 kB)
  Using c

In [3]:
# Verify imports
packages = ['langchain','openai','tiktoken','sentence_transformers','chromadb']
for p in packages:
    try:
        __import__(p)
        print(p, 'OK')
    except Exception as e:
        print(p, 'FAILED:', e)

langchain OK
openai OK
tiktoken OK


c:\Users\USER\miniconda3\envs\env_RAG\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


sentence_transformers OK
chromadb OK


In [1]:
# 1️⃣ Load environment variables (OPENAI_API_KEY must be set)
#import os

# 2️⃣ Load website content
from langchain_community.document_loaders import WebBaseLoader
loader = WebBaseLoader("https://en.wikipedia.org/wiki/Artificial_intelligence")
documents = loader.load()

c:\Users\USER\miniconda3\envs\env_RAG\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [2]:
# 3️⃣ Split text into smaller chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
docs = text_splitter.split_documents(documents)


In [3]:
OPENROUTER_API_KEY = "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"
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"


In [4]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",   # supported via OpenRouter
    api_key=OPENROUTER_API_KEY,
    base_url=OPENROUTER_BASE_URL,
)

vector_db = FAISS.from_documents(docs, embeddings)


In [5]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="openai/gpt-4o-mini",   # OpenRouter model name
    api_key=OPENROUTER_API_KEY,
    base_url=OPENROUTER_BASE_URL,
    temperature=0
)

In [6]:
from langchain.chains.retrieval_qa.base import RetrievalQA

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=vector_db.as_retriever()
)


ModuleNotFoundError: No module named 'langchain.chains'

In [2]:
from langchain.chains import RetrievalQA

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=vector_db.as_retriever()
)


ModuleNotFoundError: No module named 'langchain.chains'

In [3]:
pip install langchain.chains

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement langchain.chains (from versions: none)
ERROR: No matching distribution found for langchain.chains


In [ ]:

print(qa_chain.run("What is Artificial Intelligence?"))


In [16]:
# 4️⃣ Convert text into embeddings and store in vector database
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = OpenAIEmbeddings()
vector_db = FAISS.from_documents(docs, embeddings)

OpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable

In [15]:
pip install langchain_openai

Note: you may need to restart the kernel to use updated packages.


In [ ]:



# 5️⃣ Create RAG pipeline
from langchain_openai import ChatOpenAI
from langchain.chains import RetrievalQA

llm = ChatOpenAI(temperature=0)
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=vector_db.as_retriever()
)

# 6️⃣ Ask questions from the website
query = "What is Artificial Intelligence?"
answer = qa_chain.run(query)
print(answer)
